# YOLO11n · RUOD-R · Kaggle 训练

对应原文件：`scripts/yolo11n_e100_i640.py`。训练轮数 **100**，图像尺寸 **640**，batch **16**。

本 Notebook 已内置台账、FPS 函数，导入此 `.ipynb` 并添加 Kaggle 数据集即可运行，无需上传 `scripts`。数据集已检查，此处直接使用。

1. 在 Kaggle 导入 Notebook，选择 GPU 加速器并添加 `microning/ruod-r` 数据集。
2. 数据路径已设置为 `/kaggle/input/datasets/microning/ruod-r/RUOD-R-YOLO-Split`。
3. 使用现有 Python 3.13.15、PyTorch 2.11.0+cu128、Ultralytics 8.4.163 环境；不会自动重新安装。
4. 默认使用 GPU 0（单张 Tesla T4），依次执行各单元格。首次下载预训练权重需要开启 Internet；也可将权重添加到 Input 并修改 `PRETRAINED_WEIGHTS`。

结果写入 `/kaggle/working/underwater_object_detection`。数据划分（由你提供）：train 8608、val 956、test 4134；不重新扫描验证。
`patience=10` 可能使训练早于设定轮数结束。显存不足时将 `BATCH_SIZE` 从 16 降为 8、4 或 2，并记录调整。

## 1. 使用现有环境并导入

你提供的 Kaggle 环境已经安装 Ultralytics，直接导入使用。安装命令保持 `%pip install ultralytics`，作为注释供缺包的新环境使用。
环境快照会记录实际依赖版本，实验环境由你统一管理。

In [ ]:
# 当前 Kaggle 环境已安装 Ultralytics，直接使用现有版本。
# 仅当换用缺少该库的新环境时，取消下一行注释后执行。
# %pip install ultralytics

In [ ]:
from pathlib import Path
import csv
import json
import os
import random
import shutil
import statistics
import subprocess
import sys
import tempfile
import time
import zipfile

import torch
import ultralytics
import yaml
from ultralytics import YOLO
from IPython.display import Image, display

## 2. 设置 Kaggle 输入与输出路径

直接读取 Input 数据，生成的 YAML、模型权重、图表和台账保存到 Working。
新训练保持 `RESUME_CHECKPOINT` 和 `RESTORE_RUN_FROM` 为空；跨会话恢复方法见第 7 节。

In [ ]:
# 直接读取已添加到 Kaggle Input 的数据集，不复制数据、不重复检查标签。
DATA_ROOT = Path("/kaggle/input/datasets/microning/ruod-r/RUOD-R-YOLO-Split")
PROJECT_ROOT = Path("/kaggle/working/underwater_object_detection")
PROJECT_DIR = PROJECT_ROOT / "runs"
RUNTIME_ROOT = PROJECT_ROOT / "runtime"

# 新训练时保留两个空字符串。
# 同一会话续训：填写 /kaggle/working/.../runs/实验名/weights/last.pt。
RESUME_CHECKPOINT = ""
# 跨会话续训：将保存的整个旧实验目录添加到 Input，填写其路径。
# 例：/kaggle/input/<旧输出数据集>/underwater_object_detection/runs/<实验名>
RESTORE_RUN_FROM = ""
REQUIRE_GPU = True

RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir("/kaggle/working")

if RESTORE_RUN_FROM:
    previous_run = Path(RESTORE_RUN_FROM).resolve()
    restored_run = PROJECT_DIR / previous_run.name
    # 保留历史最佳权重和曲线，不能只恢复 last.pt。
    for relative in ("weights/last.pt", "weights/best.pt", "results.csv", "args.yaml"):
        if not (previous_run / relative).is_file():
            raise FileNotFoundError(f"旧实验目录缺少 {relative}: {previous_run}")
    if restored_run.exists():
        raise FileExistsError(
            f"目标已有实验，未覆盖: {restored_run}。若之前已恢复，请清空 RESTORE_RUN_FROM，"
            "直接把该目录下的 weights/last.pt 填入 RESUME_CHECKPOINT。"
        )
    saved_args = yaml.safe_load((previous_run / "args.yaml").read_text(encoding="utf-8"))
    original_save_dir = saved_args.get("save_dir")
    if not original_save_dir and saved_args.get("project") and saved_args.get("name"):
        original_save_dir = str(Path(saved_args["project"]) / saved_args["name"])
    if not original_save_dir or Path(original_save_dir).resolve() != restored_run.resolve():
        raise ValueError("旧实验输出路径与当前 PROJECT_DIR 不一致，请使用原 Kaggle 项目输出路径。")
    shutil.copytree(previous_run, restored_run)
    RESUME_CHECKPOINT = str(restored_run / "weights" / "last.pt")
    print("已恢复旧实验:", restored_run)

print("数据根目录:", DATA_ROOT)
print("实验结果与台账目录:", PROJECT_ROOT)

## 3. 检查训练环境

In [ ]:
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError("请在 Kaggle Notebook 设置中启用 GPU 加速器后重新运行。")
# 使用第 1 张 T4。现有轮次和显存统计按单进程设计，保持单卡运行。
DEVICE = 0 if torch.cuda.is_available() else "cpu"
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print("训练设备:", gpu_name)
if torch.cuda.is_available():
    print("GPU 数量:", torch.cuda.device_count())
    for index in range(torch.cuda.device_count()):
        print(f"GPU {index}: {torch.cuda.get_device_name(index)}, "
              f"{torch.cuda.get_device_properties(index).total_memory / 1024**3:.2f} GiB")
    print("本次训练、评估和 FPS 使用 GPU 0。")

## 4. 生成 Kaggle 数据配置

读取 Input 下 `data.yaml` 的类别配置，将路径写为 Kaggle 路径，生成 `runtime/data_kaggle.yaml`。
不修改 Input 文件，也不执行额外的图片/标签检查。Ultralytics 自身的数据加载与索引逻辑保持默认。

In [ ]:
# 仅读取 Input 的 YAML 类别信息，在 Working 生成当前平台的路径配置。
INPUT_DATA_YAML = DATA_ROOT / "data.yaml"
input_config = yaml.safe_load(INPUT_DATA_YAML.read_text(encoding="utf-8-sig"))
DATA_CONFIG = {
    "path": str(DATA_ROOT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": input_config["names"],
}
if "nc" in input_config:
    DATA_CONFIG["nc"] = input_config["nc"]
DATA_YAML = RUNTIME_ROOT / "data_kaggle.yaml"
DATA_YAML.write_text(yaml.safe_dump(DATA_CONFIG, allow_unicode=True, sort_keys=False), encoding="utf-8")
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
print(DATA_YAML.read_text(encoding="utf-8"))

## 5. 训练参数

按当前实验设置使用 `BATCH_SIZE=16`、`WORKERS=4`，其余默认值沿用对应训练脚本。断点续训也使用这里设置的 batch 和 workers；若调整它们，请记录实验差异。
测试集用于最终评估，不用于选择模型或调参。
`PRETRAINED_WEIGHTS` 默认由 Ultralytics 自动下载；离线时改为已添加到 Input 的 `.pt` 文件完整路径。首次完整运行建议开启 Internet，以便获取预训练及库内部检查所需的资源。

In [ ]:
EXPERIMENT_NAME = 'yolo11n_epoch100_imgsz640'
MODEL_NAME = 'YOLO11n'
PRETRAINED_WEIGHTS = 'yolo11n.pt'
DATASET_NAME = 'RUOD-R'
EPOCHS = 100
BATCH_SIZE = 16
IMG_SIZE = 640
WORKERS = 4
SEED = 42
PATIENCE = 10
OPTIMIZER = 'auto'
SAVE_SAMPLE_PREDICTIONS = True
SAMPLE_PREDICTION_COUNT = 20
PREDICTION_CONF = 0.25

## 6. 内置实验辅助函数

来自 `experiment_ledger.py`、`inference_benchmark.py` 和当前训练脚本；不需要额外模块文件。
台账字段与原项目保持一致。

In [ ]:
FIELDS = ['experiment_name', 'model', 'dataset', 'epochs', 'actual_epochs', 'best_epoch', 'batch', 'imgsz', 'optimizer', 'seed', 'parameters_M', 'gflops', 'gpu_name', 'gpu_peak_memory_gib', 'val_precision', 'val_recall', 'val_mAP50', 'val_mAP50_95', 'test_precision', 'test_recall', 'test_mAP50', 'test_mAP50_95', 'train_minutes', 'fps_mean', 'fps_std']

def save_experiment_record(record, project_root):

    save_dir = Path(project_root) / "records"
    save_dir.mkdir(parents=True, exist_ok=True)

    csv_path = save_dir / "experiment_ledger.csv"

    is_new = (
        not csv_path.exists()
        or csv_path.stat().st_size == 0
    )

    # 检查已有台账表头
    if not is_new:
        with csv_path.open(
            "r", encoding="utf-8-sig", newline=""
        ) as f:
            header = next(csv.reader(f), [])

        if header != FIELDS:
            raise ValueError(
                "现有台账字段不一致，请先备份旧 CSV。"
            )

    # 追加一条实验记录
    with csv_path.open(
        "a",
        newline="",
        encoding="utf-8-sig" if is_new else "utf-8"
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=FIELDS
        )

        if is_new:
            writer.writeheader()

        writer.writerow({
            field: record.get(field, "")
            for field in FIELDS
        })

    print(f"\n实验台账已保存: {csv_path}")

In [ ]:
def print_metrics(metrics, split_name):

    print(f"\n====== {split_name} 评估结果 ======")

    print(f"Precision:     {metrics.box.mp:.4f}")
    print(f"Recall:        {metrics.box.mr:.4f}")
    print(f"mAP@0.5:       {metrics.box.map50:.4f}")
    print(f"mAP@0.5:0.95:  {metrics.box.map:.4f}")

def save_sample_predictions(model, run_dir):

    if not SAVE_SAMPLE_PREDICTIONS:
        return 0

    print("\n========== 保存预测示例 ==========")

    # 测试集图片目录
    test_dir = DATA_ROOT / "images" / "test"

    # 获取测试集图片
    test_images = sorted(
        p for p in test_dir.rglob("*")
        if p.is_file()
        and p.suffix.lower() in IMAGE_SUFFIXES
    )

    if not test_images:
        print("测试集没有可供预测的图片。")
        return 0

    # 随机选取测试图片
    sample_count = min(
        SAMPLE_PREDICTION_COUNT,
        len(test_images)
    )

    rng = random.Random(SEED)

    sample_images = rng.sample(
        test_images,
        sample_count
    )

    saved_count = 0

    # 创建预测保存目录
    save_dir = run_dir / "sample_predictions"
    save_dir.mkdir(parents=True, exist_ok=True)
    source_file = save_dir / "sources.txt"
    source_file.write_text("\n".join(str(p.resolve()) for p in sample_images), encoding="utf-8")

    # 模型预测
    results = model.predict(
        source=str(source_file),
        batch=1,
        imgsz=IMG_SIZE,
        conf=PREDICTION_CONF,
        device=DEVICE,
        stream=True,
        save=False,
        verbose=False,
    )

    # 保存带检测框的图片
    for index, result in enumerate(results, start=1):

        image_name = Path(result.path).stem

        save_path = save_dir / (
            f"{index:02d}_{image_name}.jpg"
        )

        result.save(filename=str(save_path))

        saved_count += 1

        print(
            f"[{index}/{sample_count}] "
            f"已保存: {save_path.name}"
        )

    print(f"预测图片保存完成，共{saved_count}张。")

    return saved_count

In [ ]:
def _sync(device):
    """等待 GPU 计算完成，保证计时准确。"""

    if not torch.cuda.is_available():
        return

    if str(device).lower() == "cpu":
        return

    if isinstance(device, int) or str(device).isdigit():
        torch.cuda.synchronize(int(device))
    else:
        torch.cuda.synchronize(device)

def benchmark_fps(
    model,
    test_dir,
    imgsz=640,
    device=0,
    batch=1,
    warmup=50,
    repeats=3,
):
    """
    测量完整测试集的端到端 FPS。

    包含：图片读取、解码、预处理、模型推理、后处理。
    不包含：模型加载、预热、结果可视化和文件保存。

    返回：
        inference_fps: FPS 均值
        inference_fps_std: 多次测试的标准差
    """

    test_dir = Path(test_dir)

    if not test_dir.is_dir():
        raise FileNotFoundError(
            f"测试集目录不存在: {test_dir}"
        )

    image_paths = sorted(
        p for p in test_dir.rglob("*")
        if p.is_file()
        and p.suffix.lower() in IMAGE_SUFFIXES
    )

    if not image_paths:
        raise RuntimeError(
            f"测试集没有图片: {test_dir}"
        )

    if batch < 1 or warmup < 1 or repeats < 1:
        raise ValueError(
            "batch、warmup、repeats 必须大于 0。"
        )

    sources = [str(p) for p in image_paths]

    # 文件清单走文件加载器，使 batch 生效；避免全量图片列表驻留内存。
    with tempfile.TemporaryDirectory(prefix="fps_sources_") as temporary_dir:
        source_file = Path(temporary_dir) / "test_images.txt"
        source_file.write_text("\n".join(str(p.resolve()) for p in image_paths), encoding="utf-8")
        print("\n========== FPS 性能测试 ==========")
        print(f"测试集图片数: {len(sources)}")
        print(f"输入尺寸: {imgsz}")
        print(f"Batch Size: {batch}")
        print(f"预热次数: {warmup}")
        print(f"正式测试次数: {repeats}")

        # --------------------------------------
        # 1. 预热，不计入 FPS
        # --------------------------------------

        print("正在预热模型...")

        for _ in range(warmup):
            results = model.predict(
                source=str(image_paths[0]),
                imgsz=imgsz,
                batch=1,
                device=device,
                half=False,
                stream=True,
                save=False,
                verbose=False,
            )

            # 必须消费生成器，才能真正执行预测
            for _ in results:
                pass

        _sync(device)

        # --------------------------------------
        # 2. 重复测试完整测试集
        # --------------------------------------

        fps_values = []

        for repeat in range(repeats):

            _sync(device)
            start = time.perf_counter()

            processed = 0

            results = model.predict(
                source=str(source_file),
                imgsz=imgsz,
                batch=batch,
                device=device,
                half=False,
                stream=True,
                save=False,
                verbose=False,
            )

            for _ in results:
                processed += 1

            # 确保 GPU 运算完成后再停止计时
            _sync(device)

            elapsed = time.perf_counter() - start

            if processed != len(sources):
                raise RuntimeError(
                    f"预期处理 {len(sources)} 张图片，"
                    f"实际处理 {processed} 张。"
                )

            fps = processed / elapsed
            fps_values.append(fps)

            print(
                f"第 {repeat + 1}/{repeats} 次: "
                f"{fps:.2f} FPS"
            )

        # --------------------------------------
        # 3. 汇总结果
        # --------------------------------------

        fps_mean = statistics.mean(fps_values)

        fps_std = (
            statistics.stdev(fps_values)
            if len(fps_values) > 1
            else 0.0
        )

        print("\n========== FPS 测试结果 ==========")
        print(f"FPS Mean: {fps_mean:.2f}")
        print(f"FPS Std:  {fps_std:.2f}")

        return fps_mean, fps_std

## 7. 加载模型（新训练或恢复训练）

同一会话恢复：将 `RESUME_CHECKPOINT` 设为 Working 中同一次实验的 `weights/last.pt`，重新执行前面的准备步骤。
跨会话恢复：先将已保存的整个旧实验目录添加到 Input（包含 `weights/last.pt`、`weights/best.pt`、`results.csv` 和 `args.yaml`），在第 2 节填写 `RESTORE_RUN_FROM`。代码将它复制回原 Working 输出位置并自动设置续训路径；已有目标目录不会被覆盖。
恢复只适用于尚未完成且保留优化器状态的训练。使用同一个 Notebook、模型和输出根目录；batch 和 workers 使用当前参数，其余训练设置沿用断点。恢复整个实验保留历史曲线及最佳权重。旧的全局 `records/experiment_ledger.csv` 如需合并，请另行保存与整理。
续训后无法从新建的 early-stopping 对象可靠还原全程最佳轮次，因此台账的 `best_epoch` 留空；最佳权重仍以 `best.pt` 为准。

In [ ]:
if RESUME_CHECKPOINT:
    checkpoint = Path(RESUME_CHECKPOINT)
    if not checkpoint.is_file():
        raise FileNotFoundError(checkpoint)
    if not checkpoint.resolve().is_relative_to(PROJECT_DIR.resolve()):
        raise ValueError("请先用 RESTORE_RUN_FROM 将整个旧实验恢复到 Working，再从其中的 last.pt 续训。")
    model = YOLO(str(checkpoint))
    if model.ckpt.get("epoch", -1) < 0 or model.ckpt.get("optimizer") is None:
        raise ValueError("该权重没有可恢复的训练状态。已完成的实验请作为新训练开始，勿使用 resume。")
    checkpoint_args = model.ckpt.get("train_args") or {}
    expected_run = checkpoint.resolve().parent.parent
    checkpoint_save_dir = checkpoint_args.get("save_dir")
    if not checkpoint_save_dir and checkpoint_args.get("project") and checkpoint_args.get("name"):
        checkpoint_save_dir = str(Path(checkpoint_args["project"]) / checkpoint_args["name"])
    if not checkpoint_save_dir or Path(checkpoint_save_dir).resolve() != expected_run:
        raise ValueError("断点保存的输出目录与恢复位置不一致，请恢复到原 Kaggle 输出路径。")
else:
    print("\n========== 加载模型 ==========")

    model = YOLO(PRETRAINED_WEIGHTS)
    model.info()

## 8. 开始训练并统计资源

每轮保存的 `last.pt` 位于 Working；会话结束前请保存 Notebook 版本及输出，或下载结果，不能把 Working 当作跨会话的永久磁盘。
重复执行新训练会创建带编号的实验目录。续训时耗时/显存仅统计本次会话，累计轮数以训练器为准；环境记录会注明续训来源。

In [ ]:
print("\n========== 开始训练 ==========")

# 重置 GPU 显存峰值统计
if torch.cuda.is_available():
    torch.cuda.synchronize(DEVICE)
    torch.cuda.reset_peak_memory_stats(DEVICE)

# 开始计时
train_start = time.perf_counter()

# 执行训练
if RESUME_CHECKPOINT:
    model.train(resume=True, data=str(DATA_YAML), device=DEVICE, batch=BATCH_SIZE, workers=WORKERS)
else:
    model.train(
        data=str(DATA_YAML),

        epochs=EPOCHS,
        batch=BATCH_SIZE,
        imgsz=IMG_SIZE,

        device=DEVICE,
        workers=WORKERS,

        seed=SEED,
        deterministic=True,

        patience=PATIENCE,
        optimizer=OPTIMIZER,

        project=str(PROJECT_DIR),
        name=EXPERIMENT_NAME,
        exist_ok=False,

        save=True,
        plots=True,
        verbose=True,
    )

# 确保 GPU 操作完成
if torch.cuda.is_available():
    torch.cuda.synchronize(DEVICE)

# 训练耗时（分钟）
train_minutes = (
    time.perf_counter() - train_start
) / 60

if torch.cuda.is_available():

    gpu_peak_memory_gib = (
        torch.cuda.max_memory_reserved(DEVICE)
        / (1024 ** 3)
    )

else:
    gpu_peak_memory_gib = None

print("\n========== 训练资源统计 ==========")

print(f"训练耗时: {train_minutes:.2f} min")

if gpu_peak_memory_gib is not None:
    print(
        f"训练峰值显存: "
        f"{gpu_peak_memory_gib:.2f} GiB"
    )

trainer = model.trainer

# 实际完成的训练轮数
actual_epochs = int(trainer.epoch) + 1

# 最佳 Epoch
stopper = getattr(trainer, "stopper", None)

best_epoch = getattr(
    stopper, "best_epoch", None
)

if best_epoch is not None and best_epoch > 0:
    best_epoch = int(best_epoch)
else:
    best_epoch = ""

# 获取实际优化器
optimizer_object = getattr(
    trainer, "optimizer", None
)

if optimizer_object is not None:
    optimizer_name = type(
        optimizer_object
    ).__name__
else:
    optimizer_name = OPTIMIZER

print("\n========== 训练轮次信息 ==========")

print(f"设定 Epoch: {EPOCHS}")
print(f"实际 Epoch: {actual_epochs}")
if RESUME_CHECKPOINT:
    best_epoch = ""  # 恢复训练未还原 stopper 全程状态，避免误报最佳轮次。
print(f"最佳 Epoch: {best_epoch}")
print(f"实际优化器: {optimizer_name}")

# 续训时报告权重中的实际设置，避免台账使用 Notebook 中过期的参数。
EPOCHS = int(trainer.args.epochs)
BATCH_SIZE = int(trainer.args.batch)
IMG_SIZE = int(trainer.args.imgsz)
SEED = int(trainer.args.seed)

## 9. 加载最佳权重并统计模型复杂度

In [ ]:
print("\n========== 加载最佳模型 ==========")

best_path = Path(trainer.best).resolve()

if not best_path.is_file():
    raise FileNotFoundError(
        f"最佳模型权重不存在: {best_path}"
    )

run_dir = Path(trainer.save_dir).resolve()

print(f"最佳模型路径: {best_path}")

best_model = YOLO(str(best_path))

print("\n========== 模型复杂度 ==========")

# 参数量（百万）
parameters_M = (
        sum(
            p.numel()
            for p in best_model.model.parameters()
        ) / 1e6
)

# 计算 GFLOPs
# 注意：verbose=True 才返回模型统计信息
model_info = best_model.info(
    imgsz=IMG_SIZE,
    verbose=True
)

if (
    model_info is not None
    and len(model_info) >= 4
):
    gflops = model_info[3]
else:
    gflops = None

# GFLOPs 为 0 可能表示分析失败
if gflops is not None and gflops > 0:
    gflops = round(float(gflops), 2)
else:
    gflops = ""

print(f"模型参数量: {parameters_M:.3f} M")
print(f"GFLOPs: {gflops}")

In [ ]:
# 将实际依赖、数据配置和硬件信息随每次实验保存，便于复现。
environment = {
    "python": sys.version, "torch": torch.__version__,
    "ultralytics": ultralytics.__version__, "cuda": torch.version.cuda,
    "gpu": gpu_name, "data_root": str(DATA_ROOT),
    "platform": "kaggle", "device": DEVICE,
    "visible_gpu_count": torch.cuda.device_count(), "training_gpu_count": 1 if torch.cuda.is_available() else 0,
    "resume_checkpoint": RESUME_CHECKPOINT,
    "resource_statistics_scope": "current_session",
}
(run_dir / "environment.json").write_text(json.dumps(environment, ensure_ascii=False, indent=2), encoding="utf-8")
(run_dir / "requirements-freeze.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True), encoding="utf-8")
shutil.copy2(DATA_YAML, run_dir / "data_kaggle.yaml")
if "MODEL_CONFIG" in globals():
    shutil.copy2(MODEL_CONFIG, run_dir / MODEL_CONFIG.name)

## 10. 验证集评估

In [ ]:
print("\n========== 验证集评估 ==========")

val_metrics = best_model.val(
    data=str(DATA_YAML),
    split="val",

    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=WORKERS,

    plots=True,

    project=str(run_dir),
    name="validation",
    exist_ok=False,
)

print_metrics(val_metrics, "验证集")

## 11. 测试集评估

In [ ]:
print("\n========== 测试集评估 ==========")

test_metrics = best_model.val(
    data=str(DATA_YAML),
    split="test",

    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=WORKERS,

    plots=True,

    project=str(run_dir),
    name="testing",
    exist_ok=False,
)

print_metrics(test_metrics, "测试集")

## 12. 保存并展示预测示例

In [ ]:
save_sample_predictions(
    model=best_model,
    run_dir=run_dir
)

for preview in sorted((run_dir / "sample_predictions").glob("*.jpg"))[:4]:
    display(Image(filename=str(preview), width=640))

## 13. 测量端到端 FPS

使用全部测试图片，预热 50 次、测量 3 次；包含读取、解码、预处理、推理和后处理，排除结果保存。
Notebook 使用文件清单确保 batch=1 生效，避免原脚本的图片列表被一次性加载；因此 FPS 与原脚本的列表输入测量值不直接等价。
FPS 受 GPU 型号、磁盘、依赖版本影响，跨设备结果不宜直接比较。本步骤可能需要较长时间。

In [ ]:
fps_mean, fps_std = benchmark_fps(
    model=best_model,
    test_dir=DATA_ROOT / "images" / "test",
    imgsz=IMG_SIZE,
    device=DEVICE,
    batch=1,
    warmup=50,
    repeats=3,
)

## 14. 保存实验台账和查看结果

台账追加到 `PROJECT_ROOT/records/experiment_ledger.csv`；重复执行本单元格会追加一行。
若 FPS 步骤失败，可先将 `fps_mean = fps_std = float("nan")` 再保存，台账会明确记录缺失值。

In [ ]:
print("\n========== 整理实验台账 ==========")

record = {

    # ===== 实验信息 =====
    "experiment_name": run_dir.name,
    "model": MODEL_NAME,
    "dataset": DATASET_NAME,

    # ===== 训练参数 =====
    "epochs": EPOCHS,
    "actual_epochs": actual_epochs,
    "best_epoch": best_epoch,

    "batch": BATCH_SIZE,
    "imgsz": IMG_SIZE,
    "optimizer": optimizer_name,
    "seed": SEED,

    # ===== 模型复杂度 =====
    "parameters_M": round(parameters_M, 3),
    "gflops": gflops,

    # ===== 硬件资源 =====
    "gpu_name": gpu_name,

    "gpu_peak_memory_gib": (
        round(gpu_peak_memory_gib, 2)
        if gpu_peak_memory_gib is not None
        else ""
    ),

    # ===== 验证集指标 =====
    "val_precision": round(
        float(val_metrics.box.mp), 4
    ),

    "val_recall": round(
        float(val_metrics.box.mr), 4
    ),

    "val_mAP50": round(
        float(val_metrics.box.map50), 4
    ),

    "val_mAP50_95": round(
        float(val_metrics.box.map), 4
    ),

    # ===== 测试集指标 =====
    "test_precision": round(
        float(test_metrics.box.mp), 4
    ),

    "test_recall": round(
        float(test_metrics.box.mr), 4
    ),

    "test_mAP50": round(
        float(test_metrics.box.map50), 4
    ),

    "test_mAP50_95": round(
        float(test_metrics.box.map), 4
    ),

    # ===== 训练时间 =====
    "train_minutes": round(
        train_minutes, 2
    ),

    # ===== FPS =====
    "fps_mean": round(
        fps_mean, 3
    ),

    "fps_std": round(
        fps_std, 3
    ),
}

save_experiment_record(
    record=record,
    project_root=PROJECT_ROOT
)

print("\n====================================")
print(f"{MODEL_NAME} 实验全部完成！")
print("====================================")

print(f"最佳模型: {best_path}")
print(f"最佳 Epoch: {best_epoch}")
print(f"模型参数量: {parameters_M:.3f} M")
print(f"GFLOPs: {gflops}")
print(f"训练耗时: {train_minutes:.2f} min")

print(f"实验目录: {run_dir}")

In [ ]:
for filename in ("results.png", "confusion_matrix.png"):
    path = run_dir / filename
    if path.is_file():
        display(Image(filename=str(path), width=900))
print("训练目录:", run_dir)
print("最佳权重:", best_path)
print("实验台账:", PROJECT_ROOT / "records" / "experiment_ledger.csv")

## 15. 保存输出与可选打包

在 Kaggle 保存包含输出的 Notebook 版本，确认输出文件已保存后，再从版本的 Output 页面下载。
也可将下面开关设为 `True`，把当前实验和本次会话的台账打包到 `/kaggle/working`；该 ZIP 可能较大，默认不生成。

In [ ]:
EXPORT_RESULTS_ZIP = False
print("本次实验目录:", run_dir)
print("最佳权重:", best_path)
print("请在 Kaggle 保存版本及输出，或下载需要的文件。")
if EXPORT_RESULTS_ZIP:
    archive_path = Path("/kaggle/working") / f"{run_dir.name}_results.zip"
    with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(run_dir.rglob("*")):
            if path.is_file():
                archive.write(path, arcname=str(path.relative_to(PROJECT_ROOT)))
        ledger_path = PROJECT_ROOT / "records" / "experiment_ledger.csv"
        if ledger_path.is_file():
            archive.write(ledger_path, arcname="records/experiment_ledger.csv")
    print("结果压缩包:", archive_path)

## 运行提示与参考

- 每个 Notebook 可独立运行；不要在同一内核中交错执行多个实验的单元格。
- 会话重启后，重新执行导入、路径设置及 YAML 生成。若需要续训，使用已保存的完整实验目录恢复。
- 当前按单张 T4 运行，另一张 GPU 不参与训练。现有训练器轮次、优化器和显存统计依赖单进程，不能仅将 `DEVICE` 改成 `[0, 1]`。
- 本文件未包含训练输出，未在当前机器启动完整训练。训练方案来源于脚本，batch=16、workers=4 按当前要求设置。

官方参考：[Kaggle Notebook 文档](https://www.kaggle.com/docs/notebooks) · [Ultralytics 训练与恢复](https://docs.ultralytics.com/modes/train/) · [训练器实现](https://github.com/ultralytics/ultralytics/blob/v8.4.163/ultralytics/engine/trainer.py)